In [1]:
import os
import re
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from collections import Counter

from sklearn.model_selection import train_test_split

sns.set_theme(style="whitegrid")

RANDOM_STATE = 42

In [2]:
# 1. Get the directory of the current script
SCRIPT_DIR = Path.cwd()

# 2. Point to your data folder relative to this script
DATA_DIR = SCRIPT_DIR / "../data/images/images"

def get_image_dataset(foldername, DATA_DIR=DATA_DIR):
    flip = DATA_DIR/foldername/"flip"
    notFlip = DATA_DIR/foldername/"notflip"

    flipping_images = list(flip.glob("*"))
    not_flipping_images = list(notFlip.glob("*"))

    return flipping_images, not_flipping_images

In [3]:
def extract_image_metadata(image_path, label):
    image = cv2.imread(str(image_path))

    if image is None:
        return None

    height, width, channels = image.shape

    # Extract video ID and frame number
    filename = image_path.stem

    match = re.match(r"(.+?)_(\d+)$", filename)

    if match:
        video_id = match.group(1)
        frame_number = int(match.group(2))
    else:
        video_id = None
        frame_number = None

    return {
        "image_path": str(image_path),
        "filename": image_path.name,
        "video_id": video_id,
        "frame_number": frame_number,
        "label": label,
        "width": width,
        "height": height,
        "channels": channels,
        "file_size_kb": image_path.stat().st_size / 1024
    }

In [4]:
def create_image_metadata(image_paths, label, metadata):    
    for image_path in image_paths:
        result = extract_image_metadata(image_path, label=label)

        if result is not None:
            metadata.append(result)

    return metadata

In [5]:
def create_dataframe(folder_name):
    flip_images, notFlip_images = get_image_dataset(foldername=folder_name)

    metadata = []
    metadata = create_image_metadata(image_paths=flip_images, label=1, metadata=metadata)
    metadata = create_image_metadata(image_paths=notFlip_images, label=0, metadata=metadata)
    return pd.DataFrame(metadata)


In [6]:
df = create_dataframe(folder_name="training")
print(df.head())


                                          image_path            filename  \
0  c:\Users\Shane\Desktop\Other\vscode\Github Pro...  0001_000000010.jpg   
1  c:\Users\Shane\Desktop\Other\vscode\Github Pro...  0001_000000011.jpg   
2  c:\Users\Shane\Desktop\Other\vscode\Github Pro...  0001_000000012.jpg   
3  c:\Users\Shane\Desktop\Other\vscode\Github Pro...  0001_000000013.jpg   
4  c:\Users\Shane\Desktop\Other\vscode\Github Pro...  0001_000000014.jpg   

  video_id  frame_number  label  width  height  channels  file_size_kb  
0     0001            10      1   1080    1920         3    322.095703  
1     0001            11      1   1080    1920         3    322.178711  
2     0001            12      1   1080    1920         3    325.963867  
3     0001            13      1   1080    1920         3    331.750977  
4     0001            14      1   1080    1920         3    328.498047  


In [7]:
test_df = create_dataframe(folder_name="testing")
print(test_df.head())

                                          image_path            filename  \
0  c:\Users\Shane\Desktop\Other\vscode\Github Pro...  0001_000000020.jpg   
1  c:\Users\Shane\Desktop\Other\vscode\Github Pro...  0002_000000012.jpg   
2  c:\Users\Shane\Desktop\Other\vscode\Github Pro...  0002_000000013.jpg   
3  c:\Users\Shane\Desktop\Other\vscode\Github Pro...  0002_000000015.jpg   
4  c:\Users\Shane\Desktop\Other\vscode\Github Pro...  0002_000000017.jpg   

  video_id  frame_number  label  width  height  channels  file_size_kb  
0     0001            20      1   1080    1920         3    352.241211  
1     0002            12      1   1080    1920         3    292.631836  
2     0002            13      1   1080    1920         3    300.593750  
3     0002            15      1   1080    1920         3    292.907227  
4     0002            17      1   1080    1920         3    300.518555  


In [8]:
# Create a unique identifier for each video
df["video_group"] = (df["label"].astype(str) + "_" + df["video_id"].astype(str))

# Create one record per unique video
video_df = (
    df[["video_group", "label"]]
    .drop_duplicates(subset="video_group")
    .reset_index(drop=True)
)

print(video_df["label"].value_counts())

label
1    65
0    52
Name: count, dtype: int64


In [9]:
#split train/ validation dataset

# split: 80% training, 20% validation
train_videos, validation_videos = train_test_split(
    video_df,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=video_df["label"]
)

# Get unique video groups for each dataset and convert into a Python set
train_groups = set(train_videos["video_group"])
validation_groups = set(validation_videos["video_group"])

# Assign all frames belonging to each video
train_df = df[df["video_group"].isin(train_groups)].copy()

validation_df = df[df["video_group"].isin(validation_groups)].copy()

In [10]:
# Ensure no videos appear in both datasets
train_groups = set(train_df["video_group"])
validation_groups = set(validation_df["video_group"])

assert train_groups.isdisjoint(validation_groups), (
    "Data leakage detected: videos appear in both datasets."
)

# Display dataset sizes
print(f"Training images: {len(train_df)}")
print(f"Validation images: {len(validation_df)}")

print(f"Training videos: {len(train_groups)}")
print(f"Validation videos: {len(validation_groups)}")

# Check class distributions
print("\nTraining class distribution:")
print(train_df["label"].value_counts(normalize=True))

print("\nValidation class distribution:")
print(validation_df["label"].value_counts(normalize=True))

Training images: 1871
Validation images: 521
Training videos: 93
Validation videos: 24

Training class distribution:
label
0    0.520577
1    0.479423
Name: proportion, dtype: float64

Validation class distribution:
label
1    0.508637
0    0.491363
Name: proportion, dtype: float64


In [16]:
from pathlib import Path

SCRIPT_DIR = Path.cwd()

SPLIT_DIR = SCRIPT_DIR / "../data/dataset"
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

train_df.to_csv(
    SPLIT_DIR / "train.csv",
    index=False
)

validation_df.to_csv(
    SPLIT_DIR / "validation.csv",
    index=False
)

test_df.to_csv(
    SPLIT_DIR / "test.csv",
    index=False
)